In [6]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.chains.query_constructor.base import AttributeInfo
from langchain_community.document_loaders import PyPDFLoader, TextLoader, BSHTMLLoader
from langchain.retrievers import BM25Retriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

Define the Manual RRF Function

In [7]:
def reciprocal_rank_fusion(retriever_results, k=60, top_n=5):
    """
    Combine multiple ranked lists of documents using Reciprocal Rank Fusion.

    Args:
        retriever_results: List of lists, each a ranked list of Document objects.
        k: constant to control rank influence (default 60).
        top_n: number of top documents to return after fusion.

    Returns:
        List of Document objects sorted by fused scores (descending).
    """
    fused_scores = {}
    doc_map = {}

    for ranked_list in retriever_results:
        for rank, doc in enumerate(ranked_list, start=1):
            key = doc.page_content.strip()
            score = 1 / (k + rank)
            if key in fused_scores:
                fused_scores[key] += score
            else:
                fused_scores[key] = score
                doc_map[key] = doc

    sorted_keys = sorted(fused_scores, key=fused_scores.get, reverse=True)
    return [doc_map[key] for key in sorted_keys[:top_n]]

Load Documents and Create Chunks

In [8]:
files = [
    ('../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf', 'pdf'),
    ('../../04_data_ingestion_document_processing/data/crop_disease.pdf', 'pdf'),
    ('../../04_data_ingestion_document_processing/data/agriculture.html', 'html'),
    ('../../04_data_ingestion_document_processing/data/agriculture.txt', 'txt'),
]

all_docs = []
for path, ftype in files:
    if ftype == 'pdf':
        loader = PyPDFLoader(path)
    elif ftype == 'html':
        loader = BSHTMLLoader(path, open_encoding='utf-8', bs_kwargs={'features': 'html.parser'})
    elif ftype == 'txt':
        loader = TextLoader(path, encoding='utf-8')
    else:
        continue
    all_docs.extend(loader.load())

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(all_docs)

for i, chunk in enumerate(chunks):
    source = chunk.metadata.get('source', '')
    file_name = source.split('\\')[-1] if '\\' in source else source.split('/')[-1]
    chunk.metadata['file_name'] = file_name
    chunk.metadata['doc_type'] = (
        'pdf' if file_name.endswith('.pdf')
        else 'html' if file_name.endswith('.html')
        else 'txt'
    )
    chunk.metadata['language'] = 'English'
    chunk.metadata['chunk_id'] = f'{file_name}_{i+1:03d}'

print(f'Loaded {len(all_docs)} docs, created {len(chunks)} chunks.')

Loaded 37 docs, created 185 chunks.


Create Vector Store and Base Retriever

In [10]:
embeddings = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=None
)

base_retriever = vectorstore.as_retriever(search_kwargs={'k': 4})
print('Base retriever ready.')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Base retriever ready.


Create the Other Retrievers

In [13]:
# BM25 retriever (keyword-based)
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 4

# Shared LLM
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# MultiQueryRetriever
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm
)

# SelfQueryRetriever
metadata_field_info = [
    AttributeInfo(name='source', description='The file path of the document. Contains keywords like "nigeria_health" or "crop_disease" or "agriculture".', type='string'),
    AttributeInfo(name='doc_type', description='The type of document: pdf, html, txt', type='string'),
    AttributeInfo(name='language', description='Language of the document, e.g., English', type='string'),
]
document_content_description = 'Documents about agriculture, public health, and diseases in Nigeria'

self_query_retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    verbose=False
)

# ContextualCompressionRetriever
compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

print('All individual retrievers ready.')

All individual retrievers ready.


Create the other retrievers

In [14]:
# Shared LLM
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# MultiQueryRetriever
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm
)

# SelfQueryRetriever
metadata_field_info = [
    AttributeInfo(name='source', description='The file path of the document. Contains keywords like "nigeria_health" or "crop_disease" or "agriculture".', type='string'),
    AttributeInfo(name='doc_type', description='The type of document: pdf, html, txt', type='string'),
    AttributeInfo(name='language', description='Language of the document, e.g., English', type='string'),
]
document_content_description = 'Documents about agriculture, public health, and diseases in Nigeria'

self_query_retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    verbose=False
)

# ContextualCompressionRetriever
compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

print('✅ All individual retrievers ready.')

✅ All individual retrievers ready.


Run fusion with BM25

In [15]:
question = 'What are the common crop diseases and their control methods?'

# Get results from each retriever
base_results = base_retriever.invoke(question)
bm25_results = bm25_retriever.invoke(question)
multi_results = multi_query_retriever.invoke(question)
self_query_results = self_query_retriever.invoke(question)
compression_results = compression_retriever.invoke(question)

# Combine into list of lists
all_results = [base_results, bm25_results, multi_results, self_query_results, compression_results]

# Apply manual RRF
fused_docs = reciprocal_rank_fusion(all_results, k=60, top_n=5)

print(f'Fused top {len(fused_docs)} documents:\n')
for i, doc in enumerate(fused_docs, start=1):
    print(f'{i}. {doc.page_content[:250]}')
    print(f'   Source: {doc.metadata.get("source", "unknown")}')
    print('-' * 60)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Fused top 5 documents:

1. Common Crop Diseases and Their Control

1. Cassava Mosaic Disease
Affected Crop: Cassava
Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
Control/Cure: Use disease-free cuttings, plant resistant varieties, and remove infecte
   Source: ../../04_data_ingestion_document_processing/data/agriculture.html
------------------------------------------------------------
2. Common Crop Diseases and Control

1. Cassava Mosaic Disease
   Affected crop: Cassava
   Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
   Control: Use disease-free cuttings, plant resistant varieties, and remove infected 
   Source: ../../04_data_ingestion_document_processing/data/agriculture.txt
------------------------------------------------------------
3. • The major factors responsible for crop 
diseases are fungi, bacteria, viruses and 
nematodes. 
• Others are nutrient deficiencies and air 
pollutants. 
• The severity of crop diseases can 

In [17]:
question = 'What are the common crop diseases and animal diseases and their control methods?'

# Get results from each retriever
base_results = base_retriever.invoke(question)
bm25_results = bm25_retriever.invoke(question)
multi_results = multi_query_retriever.invoke(question)
self_query_results = self_query_retriever.invoke(question)
compression_results = compression_retriever.invoke(question)

# Combine into list of lists
all_results = [base_results, bm25_results, multi_results, self_query_results, compression_results]

# Apply manual RRF
fused_docs = reciprocal_rank_fusion(all_results, k=60, top_n=5)

print(f'Fused top {len(fused_docs)} documents:\n')
for i, doc in enumerate(fused_docs, start=1):
    print(f'{i}. {doc.page_content[:250]}')
    print(f'   Source: {doc.metadata.get("source", "unknown")}')
    print('-' * 60)

Fused top 5 documents:

1. Common Crop Diseases and Their Control

1. Cassava Mosaic Disease
Affected Crop: Cassava
Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
Control/Cure: Use disease-free cuttings, plant resistant varieties, and remove infecte
   Source: ../../04_data_ingestion_document_processing/data/agriculture.html
------------------------------------------------------------
2. Common Crop Diseases and Control

1. Cassava Mosaic Disease
   Affected crop: Cassava
   Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
   Control: Use disease-free cuttings, plant resistant varieties, and remove infected 
   Source: ../../04_data_ingestion_document_processing/data/agriculture.txt
------------------------------------------------------------
3. Use disease-resistant seeds and breeds.
Practise crop rotation and clean farming.
Ensure proper nutrition and clean water for animals.
Maintain hygiene in farms and pens.
Quarantine new plan